# `dfipy` Quick Start Guide

This  notebook will guide you through the basics by querying a
[small 25 million record dataset](https://www.microsoft.com/en-us/research/publication/geolife-gps-trajectory-dataset-user-guide/)
in the Data Flow Index from [General System](https://www.generalsystem.com).

`dfipy` documentation is available at [https://dfipy.docs.generalsystem.com/index.html](https://dfipy.docs.generalsystem.com/index.html).

Please refer to https://github.com/thegeneralsystem/dfipy-examples for
the most up-to-date companion documentation.

Additional resources and help are available at <https://support.generalsystem.com>.

### Let's go

### Install Dependencies

If you are using Google Colab, this will set up all required dependencies:

In [6]:
from copy import deepcopy
from datetime import datetime
from getpass import getpass
from typing import List, Optional, Union

import geopandas as gpd
import pandas as pd
import requests
from shapely.geometry import Polygon

# Google Colab setup
try:
    from google.colab import output

    output.enable_custom_widget_manager()  # allows KeplerGL map to display

    ! pip install dfipy==6.0.1 h3==3.7.6 keplergl==0.3.2

    import h3
    from dfi import Client
    from keplergl import KeplerGl

except ModuleNotFoundError:
    import h3
    from dfi import Client
    from keplergl import KeplerGl

### Next, enter you API access token

In [ ]:
api_token = getpass("Enter your API access token: ")

### Dataset description

In this tutorial we will be querying a small Geolife data set

Original source data: https://www.microsoft.com/en-us/research/publication/geolife-gps-trajectory-dataset-user-guide/

| total records	| 24.9 million |
| ------------- | -------------- |
| distinct uuids | 18.670 |

#### Hardware
- The dataset runs on a single server hosted on AWS
- The server has 2 vCPU, 8 GB ram and 1 x 75 GB NVMe SSD

#### Note: this is a shared DFI instance, and you cannot add or delete data to it.

### Connect to the DFI

In [ ]:
base_url = "https://api.prod.generalsystem.com"
dataset_id = "gs.prod-2"

dfi = Client(
    api_token=api_token,
    base_url=base_url,
    progress_bar=True,
)

### Define auxiliary methods to aggregate and analyse the datasets.

In what follow we use H3 spatial indices to aggregate large amounts of data. 

An H3 spatial index allows for efficient spatial referencing, indexing, and analysis of geospatial data at varying resolutions on a global scale. `h3_resolution` refers to the granularity of the aggregation, with higher values providing finer resolution. For more information please see: https://h3geo.org/

In [ ]:
def _aggregate_records(df_input: pd.DataFrame, hex_id: str) -> pd.DataFrame:
    return (
        df_input.groupby(hex_id)
        .agg(
            num_records=("entity_id", "count"),
            num_devices=("entity_id", "nunique"),
            first_ping=("timestamp", "min"),
            last_ping=("timestamp", "max"),
        )
        .reset_index()
    )


def add_heatmap_aggregation(
    df_records: pd.DataFrame,
    h3_resolution: int,
) -> pd.DataFrame:
    return df_records.assign(
        hex_id=lambda df: [
            h3.geo_to_h3(lat, lon, resolution=h3_resolution) for lat, lon in zip(df["latitude"], df["longitude"])
        ]
    ).pipe(_aggregate_records, "hex_id")


def build_heatmap(df_records: pd.DataFrame, h3_resolution: int) -> gpd.GeoDataFrame:
    df_binned_data = add_heatmap_aggregation(df_records=df_records, h3_resolution=h3_resolution)
    print(f"The records have been binned into {len(df_binned_data):,} hexagons")
    hex_geometries = [Polygon(h3.h3_to_geo_boundary(h3_id, geo_json=True)) for h3_id in df_binned_data.hex_id]
    gdf_binned_data = gpd.GeoDataFrame(df_binned_data, geometry=hex_geometries)

    gdf_binned_data_kepler = gdf_binned_data.copy()
    gdf_binned_data_kepler = gdf_binned_data_kepler.drop(columns=["hex_id"])
    gdf_binned_data_kepler.first_ping = gdf_binned_data_kepler.first_ping.astype(str)
    gdf_binned_data_kepler.last_ping = gdf_binned_data_kepler.last_ping.astype(str)
    gdf_binned_data_kepler = gdf_binned_data_kepler.drop(columns=["first_ping", "last_ping"])
    return gdf_binned_data_kepler

### Let's query the DFI

Home many records are there in this DFI instance?

In [ ]:
result = dfi.get.records_count(dataset_id=dataset_id)
print(f"In this DFI instance there are {result:,} records")

How many unique entities are there in this DFI instance?

In [ ]:
result = dfi.get.entities(dataset_id=dataset_id)
print(f"In this DFI instance there are {len(result):,} unique entities")
entity_id = result[0]
print(f"This is an entity identifier: '{entity_id}'")

Let's retrieve the full history of an entity

In [ ]:
entity_id = "fac817f1-bef1-4c5d-85ee-1d9fcbc61b42"
result = dfi.get.records(dataset_id=dataset_id, entities=[entity_id])
print(f"The entity '{entity_id}' has {len(result):,} records")
result

Let's show the data on a map with KeplerGL. 

KeplerGL is a powerful data visualisation software that enables users to explore and analyse large datasets in a visually engaging and intuitive manner. We will be utilising it to visualise the data retrieved to make it easier to share insights gained by DFI. For more information please see: https://kepler.gl/.

In [ ]:
def show_map(
    list_polygons: Optional[List[List[List[float]]]] = None,
    list_dfs: Optional[Union[gpd.GeoDataFrame, pd.DataFrame]] = None,
    df_records: Optional[pd.DataFrame] = None,
    map_height: int = 1200,
    config: Optional[dict] = None,
) -> KeplerGl:
    if list_polygons is None:
        list_polygons = []

    dict_polygons = {f"polygon {idx}": poly for idx, poly in enumerate(list_polygons)}

    kepler_data = {}

    if len(dict_polygons) > 0:
        kepler_data.update(
            {
                "polygons": gpd.GeoDataFrame(
                    dict_polygons.keys(),
                    geometry=[Polygon(x) for x in dict_polygons.values()],
                )
            }
        )

    if df_records is not None:
        kepler_data.update({"records": df_records.copy()})

    if list_dfs is not None:
        for idx, df in enumerate(list_dfs):
            kepler_data.update({f"df_{idx}": df.copy()})

    if config is None:
        return KeplerGl(data=deepcopy(kepler_data), height=map_height)
    return KeplerGl(data=deepcopy(kepler_data), height=map_height, config=config)

In [ ]:
show_map(df_records=result)

Both the `records` and `records_count` methods can filter data specifying optionally:

- A list of entity ids
- A polygon (ether by name or listing its vertices' coordinates)
- A time range

As shown in the following example:

In [ ]:
area_in_beijing = [
    [116.29148523153947, 39.96222543881041],
    [116.25094715574181, 39.913696214388835],
    [116.27829123517142, 39.87467196045742],
    [116.35401330128605, 39.86072934250521],
    [116.42246910852907, 39.86498581056287],
    [116.41061363353265, 39.950792648507075],
    [116.29148523153947, 39.96222543881041],
]
result = dfi.get.records(
    dataset_id=dataset_id,
    entities=[
        "ef7755e2-118c-4dc1-9da6-83840b3dc224",
        "fac817f1-bef1-4c5d-85ee-1d9fcbc61b42",
    ],
    time_interval=(datetime(2009, 1, 1, 0, 0, 0), datetime(2010, 1, 1, 0, 0, 0)),
    polygon=area_in_beijing,
)
result

Let's plot the query polygon and results on a map:

In [ ]:
show_map(df_records=result, list_polygons=[area_in_beijing])

When a query return a lot of data, we can aggregate the results in a heatmap using the popular [H3 spatial index](https://www.uber.com/en-GB/blog/h3/). You can vary the `h3_resolution` parameter: larger numbers provide finer details and smaller numbers faster (but coarser) aggregation

In [ ]:
# time format is: (year, month, day, hour, minute, second)
result = dfi.get.records(
    dataset_id=dataset_id,
    polygon=area_in_beijing,
    time_interval=(datetime(2009, 1, 1, 0, 0, 0), datetime(2009, 2, 1, 0, 0, 0)),
)
print(f"Found {len(result):,} records")

Let's build a heatmap of the data returned and visualise it on a map

- Cells with darker colours represent areas with less density of records
- Cells with ligther colours represent areas with higher density of records

#### Load map configuration

In [ ]:
url = "https://raw.githubusercontent.com/thegeneralsystem/dfipy-examples/main/examples/kepler_config/geolife.json"
response = requests.get(url, timeout=30)
kepler_config = response.json()

#### Show the heatmap

In [ ]:
heatmap = build_heatmap(result, h3_resolution=7)
map1 = show_map(map_height=600, list_dfs=[heatmap], config=kepler_config)
map1

End of notebook